In [8]:
import pandas as pd
import polars as pl

# Precision/Recall/F1

Generate precision/recall/f1 scores for the test run of wags-llm. This could be used to improve tooling or perform prompt engineering.

### Load Data


Load in the excel files

In [17]:
results_df = pd.read_csv('2026-06-11-interaction_predictions.csv',sep=',')
context_df = pd.read_excel('../2026-06-08-test1.xlsx')

truth_df = context_df.copy()
pred_df = results_df.copy()

### Clean + Create Pairs

Description

Clean

In [19]:
truth_df["pmid"] = truth_df["pmid"].astype(str)

truth_df["drug_name"] = (
    truth_df["drug_name"]
    .fillna("")
    .str.upper()
    .str.strip()
)

truth_df["gene_name"] = (
    truth_df["gene_name"]
    .fillna("")
    .str.upper()
    .str.strip()
)

pred_df["pmid"] = pred_df["pmid"].astype(str)

pred_df["drug"] = (
    pred_df["drug"]
    .fillna("")
    .str.upper()
    .str.strip()
)

pred_df["gene"] = (
    pred_df["gene"]
    .fillna("")
    .str.upper()
    .str.strip()
)

Pairs

In [20]:
truth_pairs = set(
    zip(
        truth_df["pmid"],
        truth_df["drug_name"],
        truth_df["gene_name"],
    )
)

pred_pairs = set(
    zip(
        pred_df["pmid"],
        pred_df["drug"],
        pred_df["gene"],
    )
)

Metrics

In [21]:
tp = len(pred_pairs & truth_pairs)
fp = len(pred_pairs - truth_pairs)
fn = len(truth_pairs - pred_pairs)

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else 0
)

metrics = pd.DataFrame(
    [
        {
            "true_positives": tp,
            "false_positives": fp,
            "false_negatives": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
    ]
)

metrics

,true_positives,false_positives,false_negatives,precision,recall,f1
0,12,199,88,0.056872,0.12,0.07717


In [22]:
false_positives = pred_pairs - truth_pairs
false_negatives = truth_pairs - pred_pairs

print("FP examples:")
list(false_positives)[:20]

print("FN examples:")
list(false_negatives)[:20]

FP examples:
FN examples:


[('14503796', 'EPIRUBICIN', 'BCL2'),
 ('11090099', 'SAPONIN', 'BCL2'),
 ('10638174', 'ISOTRETINOIN', 'BCL2'),
 ('15503311', 'MULTIDRUG RESISTANCE MODULATOR', 'BCL2'),
 ('9868803', 'AMINOLEVULINIC ACID', 'BCL2'),
 ('15485790', 'SELENIUM', 'BCL2'),
 ('16749867', 'DOXORUBICIN HYDROCHLORIDE', 'BCL2'),
 ('16634840', 'C5A', 'BCL2'),
 ('11903603', 'MUC-1 ANTIGEN', 'BCL2'),
 ('15118125', 'ERLOTINIB', 'EGFR'),
 ('21159314', 'RECOMBINANT INTERFERON', 'BCL2'),
 ('16001273', 'HYPERBARIC OXYGEN', 'BCL2'),
 ('12543810', 'CISPLATIN', 'BCL2'),
 ('18199554', 'ERLOTINIB', 'EGFR'),
 ('8702992', 'PROTEASE INHIBITOR', 'BCL2'),
 ('11751483', 'MITOXANTRONE', 'BCL2'),
 ('8324744', 'FLOXURIDINE', 'BCL2'),
 ('11813903', 'NONSTEROIDAL ANTIINFLAMMATORY DRUG', 'BCL2'),
 ('20022809', 'ERLOTINIB', 'EGFR'),
 ('2538824', 'RECOMBINANT INTERLEUKIN', 'BCL2')]

In [23]:
truth_df = context_df.to_pandas() if hasattr(context_df, "to_pandas") else context_df.copy()
pred_df = results_df.to_pandas() if hasattr(results_df, "to_pandas") else results_df.copy()

# normalize
truth_df["pmid"] = truth_df["pmid"].astype(str)
truth_df["drug_name_norm"] = truth_df["drug_name"].fillna("").str.upper().str.strip()
truth_df["gene_name_norm"] = truth_df["gene_name"].fillna("").str.upper().str.strip()

pred_df["pmid"] = pred_df["pmid"].astype(str)
pred_df["drug_norm"] = pred_df["drug"].fillna("").str.upper().str.strip()
pred_df["gene_norm"] = pred_df["gene"].fillna("").str.upper().str.strip()

# unique truth and prediction triples
truth_triples = set(
    zip(truth_df["pmid"], truth_df["drug_name_norm"], truth_df["gene_name_norm"])
)

pred_triples = set(
    zip(pred_df["pmid"], pred_df["drug_norm"], pred_df["gene_norm"])
)

recalled = truth_triples & pred_triples
missed = truth_triples - pred_triples

recall = len(recalled) / len(truth_triples)

summary = pd.DataFrame([
    {
        "started_with": len(truth_triples),
        "recalled": len(recalled),
        "missed": len(missed),
        "recall": recall,
    }
])

summary

,started_with,recalled,missed,recall
0,100,12,88,0.12


In [25]:
missed_df = pd.DataFrame(
    missed,
    columns=["pmid", "drug_name", "gene_name"]
).sort_values(["pmid", "drug_name", "gene_name"])

missed_df.head(25)

,pmid,drug_name,gene_name
21,10203695,MASOPROCOL,BCL2
50,10220568,HYDROQUINONE,BCL2
57,10432288,FAS LIGAND,BCL2
27,10620631,MICELLAR PACLITAXEL,BCL2
29,10621853,ISOTRETINOIN,BCL2
2,10638174,ISOTRETINOIN,BCL2
26,10696460,4-PHENYLBUTYRIC ACID,BCL2
43,10785598,RALTITREXED,BCL2
54,10914739,CAELYX,BCL2
1,11090099,SAPONIN,BCL2


In [24]:
recalled_df = pd.DataFrame(
    recalled,
    columns=["pmid", "drug_name", "gene_name"]
).sort_values(["pmid", "drug_name", "gene_name"])

recalled_df.head(25)

,pmid,drug_name,gene_name
0,16043828,ERLOTINIB,EGFR
8,16190549,RIBAVIRIN,BCL2
6,18227510,ERLOTINIB,EGFR
2,19147750,ERLOTINIB,EGFR
10,19455431,ERLOTINIB,EGFR
5,19692684,ERLOTINIB,EGFR
7,21233402,ERLOTINIB,EGFR
9,22753918,ERLOTINIB,EGFR
1,23816963,ERLOTINIB,EGFR
3,24636847,ERLOTINIB,EGFR


In [27]:
# Make sure both are strings
results_df["pmid"] = results_df["pmid"].astype(str)
missed_df["pmid"] = missed_df["pmid"].astype(str)

# One row per PMID with candidate metadata
candidate_df = (
    results_df[
        ["pmid", "candidate_drugs", "candidate_genes", "candidate_drug_count", "candidate_gene_count"]
    ]
    .drop_duplicates(subset=["pmid"])
)

missed_analysis = missed_df.merge(
    candidate_df,
    on="pmid",
    how="left",
)

def split_candidates(value):
    if pd.isna(value) or value == "":
        return []
    return [x.strip().upper() for x in str(value).split(";") if x.strip()]

missed_analysis["candidate_drug_list"] = missed_analysis["candidate_drugs"].apply(split_candidates)
missed_analysis["candidate_gene_list"] = missed_analysis["candidate_genes"].apply(split_candidates)

missed_analysis["truth_drug_norm"] = missed_analysis["drug_name"].str.upper().str.strip()
missed_analysis["truth_gene_norm"] = missed_analysis["gene_name"].str.upper().str.strip()

missed_analysis["drug_in_candidates"] = missed_analysis.apply(
    lambda row: row["truth_drug_norm"] in row["candidate_drug_list"],
    axis=1,
)

missed_analysis["gene_in_candidates"] = missed_analysis.apply(
    lambda row: row["truth_gene_norm"] in row["candidate_gene_list"],
    axis=1,
)

missed_analysis[
    [
        "pmid",
        "drug_name",
        "gene_name",
        "candidate_drugs",
        "candidate_genes",
        "drug_in_candidates",
        "gene_in_candidates",
    ]
].head(25)

,pmid,drug_name,gene_name,candidate_drugs,candidate_genes,drug_in_candidates,gene_in_candidates
0,10203695,MASOPROCOL,BCL2,CERAMIDE;Cycloheximide;chlorhexidine gluconate...,FAS;FASLG;FXN,True,False
1,10220568,HYDROQUINONE,BCL2,NaN,NaN,False,False
2,10432288,FAS LIGAND,BCL2,calcium;paclitaxel,BAD;BAX;FASLG;FXN,False,False
3,10620631,MICELLAR PACLITAXEL,BCL2,NaN,NaN,False,False
4,10621853,ISOTRETINOIN,BCL2,NaN,NaN,False,False
5,10638174,ISOTRETINOIN,BCL2,NaN,NaN,False,False
6,10696460,4-PHENYLBUTYRIC ACID,BCL2,4-phenylbutyric acid;Therapeutic Androgen,BAX;FXN;TP53,True,False
7,10785598,RALTITREXED,BCL2,Raltitrexed;Thymidine 5'-triphosphate;raltitrexed,BAX;TBXT;TCEAL1;TP53,True,False
8,10914739,CAELYX,BCL2,NaN,NaN,False,False
9,11090099,SAPONIN,BCL2,PROPIDIUM;dexamethasone;valine,TP53,False,False
